# Addestra MyCode su Google Colab (gratis)

Usa questo notebook se la tua scheda video non ce la fa. Colab ti presta una GPU T4 da 16 GB.

1. Menu **Runtime → Cambia tipo di runtime → GPU T4**.
2. Esegui le celle una alla volta con il tasto ▶ (oppure **Runtime → Esegui tutto**).
3. Alla fine il modello finisce nel tuo Google Drive, nella cartella `MyCode`.
4. Scarica il file `.gguf` da Drive, mettilo nella cartella `finetune/mycode` di MyDevAgent sul tuo PC e lancia `finetune\mycode\crea_mycode.bat`.

Ci vogliono circa un paio d'ore. Tieni la scheda del browser aperta.

In [ ]:
# 1. Controlla di avere la GPU (deve comparire "Tesla T4")
!nvidia-smi

In [ ]:
# 2. Scarica MyDevAgent con il manuale e gli esempi di MyCode
BRANCH = "claude/project-thread-ph10kn"  # quando MyCode sarà sul branch principale: "claude/gracious-mayer-mt8l9b"
!git clone --depth 1 -b {BRANCH} https://github.com/giovannisantorofrancesco2011-arch/MyDevAgent.git mydevagent
%cd mydevagent
!wc -l finetune/mycode/data/*.jsonl

**Facoltativo.** Se con `md_to_model.py` hai insegnato a MyCode altri file, carica qui il tuo `finetune/mycode/data/generated.jsonl`: icona della cartella a sinistra → trascina il file dentro `mydevagent/finetune/mycode/data/`.

In [ ]:
# 3. Installa Unsloth (addestramento veloce e leggero)
!pip install -q unsloth pyyaml

In [ ]:
# 4. Addestra MyCode ed esporta il file .gguf per Ollama
!python finetune/train_qlora.py --config finetune/mycode/config.yaml --export-gguf

In [ ]:
# 5. Copia il modello nel tuo Google Drive (cartella MyCode)
import glob, os, shutil
from google.colab import drive
drive.mount('/content/drive')
files = sorted(glob.glob('finetune/mycode/outputs/**/*.gguf', recursive=True), key=os.path.getsize)
best = [f for f in files if 'q4_k_m' in f.lower()] or files
os.makedirs('/content/drive/MyDrive/MyCode', exist_ok=True)
shutil.copy(best[-1], '/content/drive/MyDrive/MyCode/mycode-q4_k_m.gguf')
print('Fatto: Google Drive → MyCode → mycode-q4_k_m.gguf')

## Sul tuo PC

Scarica `mycode-q4_k_m.gguf` da Google Drive e mettilo in `finetune\mycode\` dentro la cartella di MyDevAgent. Poi fai doppio clic su `finetune\mycode\crea_mycode.bat` (oppure `python finetune\mycode\crea_mycode.py`).

Provalo con `ollama run mycode`, oppure in MyDevAgent con `mydevagent -p mycode`.